In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import ndcg_score
from sklearn.model_selection import TimeSeriesSplit


In [2]:
# TCVファイルの読み込み
train_A = pd.read_csv('../000.data/train/train_A.tsv', sep="\t")
train_B = pd.read_csv('../000.data/train/train_B.tsv', sep="\t")
train_C = pd.read_csv('../000.data/train/train_C.tsv', sep="\t")
train_D = pd.read_csv('../000.data/train/train_D.tsv', sep="\t")
test_data = pd.read_csv('../000.data/test.tsv', sep="\t")


In [3]:
# trainを一つに
train_data = pd.concat([train_A, train_B, train_C, train_D], ignore_index=True)


In [4]:
# 関連度スコアの設定
def compute_relevance(row):
    if row["event_type"] == 3 and row["ad"] == 1:  # 広告経由の購入
        return 3
    elif row["event_type"] == 2:  # 広告クリック
        return 2
    elif row["event_type"] == 1:  # 詳細ページ閲覧
        return 1
    else:  # カート追加
        return 0

train_data["relevance"] = train_data.apply(compute_relevance, axis=1)


In [5]:
# タイムスタンプを数値型に変換
train_data["time_stamp"] = pd.to_datetime(train_data["time_stamp"])
train_data['time_stamp'] = train_data['time_stamp'].astype(np.int64) // 10**9  # UNIX時間に変換


In [6]:
# ユーザIDと商品IDのエンコード
user_enc = LabelEncoder()
product_enc = LabelEncoder()

train_data['user_id'] = user_enc.fit_transform(train_data['user_id'])
train_data['product_id'] = product_enc.fit_transform(train_data['product_id'])

test_data[('user_id')] = user_enc.transform(test_data['user_id'])


In [7]:
# XGBoost 用データ作成
train_X = train_data[['user_id', 'product_id', 'ad', 'time_stamp']]
train_y = train_data['relevance']


In [8]:
# 時系列交差検証
tscv = TimeSeriesSplit(n_splits=3)
model = xgb.XGBRegressor(objective='reg:squarederror')

for train_idx, val_idx in tscv.split(train_X):
    X_train, X_val = train_X.iloc[train_idx], train_X.iloc[val_idx]
    y_train, y_val = train_y.iloc[train_idx], train_y.iloc[val_idx]
    model.fit(X_train, y_train)
    predictions = model.predict(X_val)
    print("Validation NDCG:", ndcg_score([y_val], [predictions]))


Validation NDCG: 0.9859634440417109
Validation NDCG: 0.9628785633106403
Validation NDCG: 0.912953422103459


In [9]:
# 最終学習
model.fit(train_X, train_y)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [10]:
# 推薦関数
def recommend_products(user_id, top_k=22):
    user_data = train_data[train_data['user_id'] == user_id][['user_id', 'product_id', 'ad', 'time_stamp']]
    if user_data.empty:
        print(f"Warning: User {user_id} has no data.")
        return pd.DataFrame(columns=['product_id', 'rank'])
    predictions = model.predict(user_data)
    user_data['score'] = predictions
    recommendations = user_data.sort_values(by='score', ascending=False).head(top_k)
    recommendations['rank'] = range(1, len(recommendations) + 1)
    recommendations['user_id'] = user_enc.inverse_transform(recommendations['user_id'])
    recommendations['product_id'] = product_enc.inverse_transform(recommendations['product_id'])
    return recommendations[['user_id', 'product_id', 'rank']]


In [11]:
# nGCDの計算
def evaluate_ndcg():
    y_true = []
    y_score = []
    for user_id in test_data['user_id'].unique():
        if user_id in train_data['user_id'].values:
            actual = train_data[train_data['user_id'] == user_id].sort_values(by='relevance', ascending=False)['relevance'].values
            pred = recommend_products(user_id, top_k=len(actual))['rank'].values
            if len(pred) == 0:
                print(f"Warning: No predictions for user {user_id}.")
                continue
            y_true.append(actual)
            y_score.append(pred)
    return np.mean([ndcg_score([t], [s]) for t, s in zip(y_true, y_score)]) if y_true else 0.0


In [12]:
# 予測結果保存
def save_predictions():
    results = []
    for user_id in test_data['user_id'].unique():
        if user_id in train_data['user_id'].values:
            recommendations = recommend_products(user_id)
            for _, row in recommendations.iterrows():
                results.append([row['user_id'], row['product_id'], row['rank']])
    if not results:
        print("Warning: No predictions generated.")
    pd.DataFrame(results, columns=['user_id', 'product_id', 'rank']).to_csv('predictions.tsv', sep='\t', index=False, header=False)
    

In [13]:
# 推薦結果表示
print("nDCG Score:", evaluate_ndcg())


nDCG Score: 0.7719688550177269


In [14]:
save_predictions()
